# Multi-Agent Resume Formatter and ATS Keyword Checker

##Garv Agrawal
##23B0360


This is a basic LangGraph project that takes rough, unformatted resume text and cleans it up. It uses a supervisor pattern with a shared state dictionary to run two worker nodes: one for **converting the raw text into clean Markdown** formatting, and another to **scan for typical technical keywords**. It also includes a basic loop to retry the formatting if it misses standard headers on the first pass.

In [1]:
!pip install openai langgraph

In [2]:
from typing import Dict, TypedDict
from openai import OpenAI
from langgraph.graph import StateGraph, END
from getpass import getpass

#  API key
nvidia_key = "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"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_key
)

# Shared state dictionary
class ResumeState(TypedDict):
    raw_input: str
    markdown_output: str
    loop_count: int

In [3]:
# Agent 1: Formats the text to markdown
def formatter_agent(state: ResumeState) -> Dict:
    print("\n[Running Formatter Agent]")
    user_text = state["raw_input"]

    prompt = (
        "Convert the following messy resume details into structured Markdown. "
        "Use headers like ## Experience, ## Projects, and ## Skills with bullet points. "
        "Do not write an introduction, output only the markdown code.\n\n"
        f"Input:\n{user_text}"
    )

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    output_text = response.choices[0].message.content
    print("--- Formatter finished processing ---")
    return {"markdown_output": output_text}


# Agent 2: Checks technical keywords
def keyword_agent(state: ResumeState) -> Dict:
    print("\n[Running ATS Keyword Agent]")
    current_resume = state["markdown_output"]
    target_keywords = ["Python", "C++", "Data Structures", "Git", "SQL", "Machine Learning"]



    prompt = (
        f"Scan this resume for these tech keywords: {target_keywords}. "
        "Add a section at the bottom titled '### Suggested Highlights' showing what was found "
        "and suggesting where the missing skills could fit in.\n\n"
        f"Resume:\n{current_resume}"
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    output_text = response.choices[0].message.content
    print("--- Keyword agent finished processing ---")
    return {"markdown_output": output_text}

In [4]:
# Central supervisor router
def supervisor(state: ResumeState) -> str:
    print("\n[Supervisor checking state]")
    text = state.get("markdown_output", "")
    loops = state.get("loop_count", 0)

    if not text:
        print("-> State empty. Routing to format_node.")
        return "format"

    if "##" not in text and loops < 1:
        print("-> Formatting check failed (no headers). Retrying format_node.")
        state["loop_count"] = loops + 1
        return "format"

    if "Suggested" in text or "Keywords" in text:
        print("-> Tasks finished. Routing to END.")
        return "finish"

    print("-> Structured text found. Routing to keyword_node.")
    return "analyze"

In [5]:
builder = StateGraph(ResumeState)

builder.add_node("format_node", formatter_agent)
builder.add_node("keyword_node", keyword_agent)

builder.set_conditional_entry_point(
    supervisor,
    {
        "format": "format_node",
        "analyze": "keyword_node",
        "finish": END
    }
)

builder.add_conditional_edges(
    "format_node",
    supervisor,
    {
        "format": "format_node",
        "analyze": "keyword_node",
        "finish": END
    }
)

builder.add_conditional_edges(
    "keyword_node",
    supervisor,
    {
        "format": "format_node",
        "analyze": "keyword_node",
        "finish": END
    }
)

graph_app = builder.compile()
print("Graph successfully compiled.")

Graph successfully compiled.


In [6]:
print("PASTE ROUGH RESUME TEXT AND PRESS ENTER:")
user_resume_input = input()

if user_resume_input.strip():
    print(f"\n[Input Echo]\n{user_resume_input}\n")

    initial_state = {
        "raw_input": user_resume_input,
        "markdown_output": "",
        "loop_count": 0
    }

    print("Starting workflow...")
    final_output = graph_app.invoke(initial_state)

    print("\n" + "="*20 + " FINAL RESUME OUTPUT " + "="*20 + "\n")
    print(final_output["markdown_output"])
else:
    print("No text input entered.")

PASTE ROUGH RESUME TEXT AND PRESS ENTER:
Garv Agrawal Phone-Alt +91 73765 95922 | Envelope garvagrawal005@gmail.com | LINKEDIN linkedin.com/in/garvagrawal TECHNICAL SKILLS Languages : Python, C++, JavaScript, Typescript, HTML/CSS Frameworks : Django, Next.js, Angular Libraries & Tools : NumPy, Pandas, Sklearn, Matplotlib, Git, GitHub GenAI : Claude Code, Cursor, Prompt Engineering EXPERIENCE Software Developer Intern May 2025 – July 2025 Blackhill Technologies Bangalore, Karnataka • Developed an end-to-end Portfolio X-Ray Dashboard that automates the analysis of mutual fund PDFs, transforming raw financial data into comprehensive health diagnostics and benchmarking reports against top-tier market schemes. • Engineered seamless RESTful API integrations using an asynchronous polling mechanism to manage real-time data flow, ensuring smooth state transitions from document upload to final report rendering. Web Coordinator Dec. 2024 – Feb. 2025 Entrepreneurship-Cell, IIT Bombay Mumbai, Mahar